# RailTrack Analytics: Professional Train Delay Dashboard

A high-performance analytics system adapted for Jupyter Notebook using Plotly.

## Features:
- 3D Surface & Scatter Cluster Analysis
- Geographical Mapbox Integration
- Temporal Distribution & Density Modeling
- KPI Trend Tracking
- Advanced 3D Analytics (Space-Time, Congestion Towers, Vortex)

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from datetime import datetime, timedelta

# Ensure reproducible results
np.random.seed(42)

## 1. Data Generation Engine
Simulates high-fidelity train telemetry data.

In [ ]:
def load_data(n=1500):
    """Generates high-fidelity train telemetry data."""
    stations = ['Central', 'East', 'West', 'North', 'South']
    zones = ['Northern', 'Southern', 'Eastern', 'Western', 'Central']
    trains = ['Express-101', 'Silver Arrow', 'Midnight Flyer', 'Coastal Link', 'Mountain Climber']
    causes = ['Signal Failure', 'Track Maintenance', 'Weather', 'Technical Fault', 'Congestion']
    
    # Generate base data
    dates = [datetime.now() - timedelta(minutes=np.random.randint(0, 1440)) for _ in range(n)]
    
    df = pd.DataFrame({
        'Train_ID': [f'TRN-{np.random.randint(100, 999)}' for _ in range(n)],
        'Train_Name': [np.random.choice(trains) for _ in range(n)],
        'Station_Name': [np.random.choice(stations) for _ in range(n)],
        'Railway_Zone': [np.random.choice(zones) for _ in range(n)],
        'Arrival_Time': dates,
        'Scheduled_Arrival': dates, # Simplified for demo
        'Lat': np.random.uniform(20.0, 28.0, n),
        'Lng': np.random.uniform(77.0, 85.0, n),
        'Distance_KM': np.random.randint(50, 1000, n),
        'Delay_Minutes': np.random.exponential(scale=15, size=n).astype(int),
        'Cause': [np.random.choice(causes) for _ in range(n)]
    })
    
    # Derived metrics
    df['Hour'] = df['Arrival_Time'].dt.hour
    df['Time_Decimal'] = df['Hour'] + df['Arrival_Time'].dt.minute / 60.0
    df['Speed_KMH'] = np.random.normal(80, 15, n) - (df['Delay_Minutes'] * 0.5) # Speed drops with delay
    df['Speed_KMH'] = df['Speed_KMH'].clip(20, 160)
    df['On_Time'] = df['Delay_Minutes'] < 5
    
    # Add some structure to Lat/Lng based on Zone for better visualization
    zone_coords = {
        'Northern': (28.6, 77.2), 'Southern': (13.0, 80.2), 
        'Eastern': (22.5, 88.3), 'Western': (19.0, 72.8), 
        'Central': (21.1, 79.0)
    }
    
    for i, row in df.iterrows():
        base_lat, base_lng = zone_coords[row['Railway_Zone']]
        df.at[i, 'Lat'] = base_lat + np.random.normal(0, 1.5)
        df.at[i, 'Lng'] = base_lng + np.random.normal(0, 1.5)

    return df

# Load the data
df = load_data()
print(f"Loaded {len(df)} records.")
df.head()

## 2. Key Performance Indicators (KPIs)

In [ ]:
avg_delay = df['Delay_Minutes'].mean()
on_time_pct = (df['On_Time'].mean() * 100)
total_incidents = len(df)
active_trains = len(df['Train_ID'].unique())

print(f"Avg System Delay:    {avg_delay:.1f} min")
print(f"On-Time Performance: {on_time_pct:.1f}%")
print(f"Total Incidents:     {total_incidents}")
print(f"Active Trains:       {active_trains}")

## 3. Emerging Delay Clusters (Packed Bubble Chart)
Visualizing delay intensity grouped by Railway Zone.

In [ ]:
grouped = df.groupby(['Railway_Zone', 'Train_Name'])['Delay_Minutes'].mean().reset_index()
grouped = grouped[grouped['Delay_Minutes'] > 0]

fig = px.scatter(grouped, x='Railway_Zone', y='Delay_Minutes',
                 size='Delay_Minutes', color='Railway_Zone',
                 hover_name='Train_Name', text='Train_Name',
                 size_max=60, template='plotly_dark',
                 title="Emerging Delay Clusters by Zone")

fig.update_traces(marker=dict(line=dict(width=1, color='white')), textposition='top center')
fig.update_layout(xaxis_title="Railway Zone", yaxis_title="Average Delay (min)", showlegend=False, height=600)
fig.show()

## 4. Advanced 3D Analytics

### 4.1 4D Spatiotemporal Trajectories (Space-Time Cube)
Visualizing train paths through geography (X/Y) and time (Z).

In [ ]:
# Filter top 10 delayed trains for clarity
top_trains = df.groupby('Train_ID')['Delay_Minutes'].sum().nlargest(10).index
plot_df = df[df['Train_ID'].isin(top_trains)].sort_values('Arrival_Time')

fig = px.line_3d(plot_df, x='Lng', y='Lat', z='Time_Decimal', color='Train_ID',
                 title="4D Spatiotemporal Trajectories (Space-Time Cube)",
                 template='plotly_dark')

fig.update_traces(line=dict(width=4), marker=dict(size=3))
fig.update_layout(scene=dict(
    xaxis_title='Longitude', yaxis_title='Latitude', zaxis_title='Time (24h)',
    aspectmode='manual', aspectratio=dict(x=1, y=1, z=0.8)
), height=700)
fig.show()

### 4.2 3D Congestion Towers
Hexbin-style map showing accumulated delay volume by location.

In [ ]:
# Bin data spatially
df['Lat_Bin'] = df['Lat'].round(1)
df['Lng_Bin'] = df['Lng'].round(1)
grouped_congestion = df.groupby(['Lat_Bin', 'Lng_Bin']).agg({'Delay_Minutes': 'sum', 'Station_Name': 'first'}).reset_index()
grouped_congestion = grouped_congestion[grouped_congestion['Delay_Minutes'] > 50] # Filter noise

fig = go.Figure()

# Towers
fig.add_trace(go.Scatter3d(
    x=grouped_congestion['Lng_Bin'], y=grouped_congestion['Lat_Bin'], z=grouped_congestion['Delay_Minutes'],
    mode='markers',
    marker=dict(
        size=8, color=grouped_congestion['Delay_Minutes'], colorscale='Portland',
        symbol='square', opacity=0.9
    ),
    text=grouped_congestion['Station_Name'],
    name='Delay Volume'
))

# Ground Shadow
fig.add_trace(go.Scatter3d(
    x=grouped_congestion['Lng_Bin'], y=grouped_congestion['Lat_Bin'], z=[0]*len(grouped_congestion),
    mode='markers',
    marker=dict(size=4, color='black', opacity=0.2),
    hoverinfo='skip'
))

fig.update_layout(
    title="3D Congestion Towers (Delay Volume)",
    template='plotly_dark',
    scene=dict(xaxis_title='Longitude', yaxis_title='Latitude', zaxis_title='Total Delay (min)'),
    height=700
)
fig.show()

### 4.3 3D Zone-Cause Matrix
Scatter plot correlating Zone, Cause, and Delay.

In [ ]:
grouped_cause = df.groupby(['Railway_Zone', 'Cause'])['Delay_Minutes'].sum().reset_index()

fig = go.Figure(data=[go.Scatter3d(
    x=grouped_cause['Railway_Zone'],
    y=grouped_cause['Cause'],
    z=grouped_cause['Delay_Minutes'],
    mode='markers',
    marker=dict(
        size=np.sqrt(grouped_cause['Delay_Minutes']),
        color=grouped_cause['Delay_Minutes'],
        colorscale='Turbo',
        opacity=0.8
    ),
    hovertemplate='Zone: %{x}<br>Cause: %{y}<br>Delay: %{z}m<extra></extra>'
)])

fig.update_layout(
    title="3D Zone-Cause Matrix",
    template='plotly_dark',
    scene=dict(xaxis_title='Zone', yaxis_title='Cause', zaxis_title='Total Delay'),
    height=700
)
fig.show()

### 4.4 Velocity Vortex
Correlation between Speed, Distance, and Delay.

In [ ]:
fig = go.Figure(data=[go.Scatter3d(
    x=df['Distance_KM'],
    y=df['Speed_KMH'],
    z=df['Delay_Minutes'],
    mode='markers',
    marker=dict(
        size=4,
        color=df['Speed_KMH'],
        colorscale='Bluered',
        opacity=0.6
    ),
    text=df['Train_Name'],
    hovertemplate='Dist: %{x}km<br>Speed: %{y:.1f}km/h<br>Delay: %{z}m<extra></extra>'
)])

fig.update_layout(
    title="Velocity Vortex (Speed vs Distance vs Delay)",
    template='plotly_dark',
    scene=dict(xaxis_title='Distance (km)', yaxis_title='Speed (km/h)', zaxis_title='Delay (min)'),
    height=700
)
fig.show()